# Tutorial: Yield Curve con Datos Reales - bonos-lib

Análisis de yield curve usando bonos cupón cero con datos del mercado.

## 1. Instalar bonos-lib

In [ ]:
!pip install bonos-lib

## 2. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bonos_lib import YieldCurveCalculator, LinearInterpolator

## 3. Datos Reales - Bonos Cupón Cero

Datos extraídos del análisis de mercado.
- **Nominal**: 100
- **Plazos**: 0.25 a 3.5 años
- **Todos son cupón cero**

In [ ]:
# Crear datos: plazos en años y precios
bonos_reales = pd.DataFrame({
    'plazo_años': [0.25, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5],
    'precio': [98.15, 97.12, 97.18, 103.49, 99.49, 98.5, 98.35, 101.88]
})

# Convertir años a días (aproximación: 1 año = 360 días)
bonos_reales['dias'] = (bonos_reales['plazo_años'] * 360).astype(int)

print("📊 Bonos Cupón Cero (Nominal = 100):")
print(bonos_reales[['plazo_años', 'dias', 'precio']])
print(f"\nTotal de bonos: {len(bonos_reales)}")

## 4. Calcular Yield Curve

In [ ]:
# Crear calculator con datos reales
calc = YieldCurveCalculator(nominal=100, days_year=360)

# Calcular yields
yields = calc.calculate_yields(bonos_reales[['dias', 'precio']])

print("📈 Yield Curve Calculada:")
print("="*50)
yields_display = yields.copy()
yields_display['yield_pct'] = yields_display['yield'] * 100
yields_display['plazo_años'] = bonos_reales['plazo_años'].values
print(yields_display[['plazo_años', 'dias', 'precio', 'yield_pct']].to_string(index=False))

print(f"\n📊 Estadísticas:")
print(f"   Tasa mínima: {yields['yield'].min()*100:.4f}%")
print(f"   Tasa máxima: {yields['yield'].max()*100:.4f}%")
print(f"   Tasa promedio: {yields['yield'].mean()*100:.4f}%")

## 5. Visualizar Yield Curve

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(bonos_reales['plazo_años'], yields['yield'] * 100, 'o-', 
         linewidth=2.5, markersize=10, color='#2E86AB', label='Yield Curve')
plt.xlabel('Plazo (años)', fontsize=12, fontweight='bold')
plt.ylabel('Tasa de Rendimiento (%)', fontsize=12, fontweight='bold')
plt.title('Yield Curve - Bonos Cupón Cero (Datos Reales)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 6. Bootstrapping - Tasas Forward

In [ ]:
# Realizar bootstrapping
forwards = calc.bootstrap(yields)

print("🔄 Tasas Spot y Forward:")
print("="*70)
forwards_display = forwards.copy()
forwards_display['spot_rate_pct'] = forwards_display['spot_rate'] * 100
forwards_display['forward_rate_pct'] = forwards_display['forward_rate'] * 100
forwards_display['plazo_años'] = bonos_reales['plazo_años'].values
print(forwards_display[['plazo_años', 'dias', 'spot_rate_pct', 'forward_rate_pct']].round(4).to_string(index=False))

print(f"\n✅ Tasas forward calculadas correctamente")

## 7. Interpolación - Tasas Intermedias

In [ ]:
# Interpolar tasas para plazos específicos
plazos_interpolacion = [90, 180, 360, 540, 720, 900, 1080]  # en días
plazos_años = [d/360 for d in plazos_interpolacion]

print("🔍 Tasas Interpoladas para Plazos Intermedios:")
print("="*50)
print(f"{'Plazo (años)':<15} {'Plazo (días)':<15} {'Tasa (%)':<15}")
print("-"*50)

tasas_interp = {}
for plazo_d, plazo_a in zip(plazos_interpolacion, plazos_años):
    try:
        tasa = calc.interpolate(yields, plazo_d)
        tasas_interp[plazo_a] = tasa
        print(f"{plazo_a:<15.2f} {plazo_d:<15} {tasa*100:<15.4f}")
    except ValueError as e:
        print(f"{plazo_a:<15.2f} {plazo_d:<15} Fuera de rango")

## 8. Curva Interpolada Detallada

In [ ]:
# Crear curva interpolada fina
interpolator = LinearInterpolator()
min_dias = int(bonos_reales['dias'].min())
max_dias = int(bonos_reales['dias'].max())

curve_fina = interpolator.interpolate_range(yields, min_days=min_dias, max_days=max_dias, step=10)
curve_fina['plazo_años'] = curve_fina['dias'] / 360

print(f"Curva interpolada creada con {len(curve_fina)} puntos")

## 9. Visualizar Yield Curve Interpolada

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

# Datos observados
ax.scatter(bonos_reales['plazo_años'], yields['yield']*100, 
          s=150, color='#D62828', zorder=5, label='Bonos observados', edgecolors='black', linewidth=1.5)

# Curva interpolada
ax.plot(curve_fina['plazo_años'], curve_fina['yield']*100, 
       linewidth=2.5, color='#2E86AB', alpha=0.8, label='Interpolación lineal')

# Tasas forward
ax.plot(bonos_reales['plazo_años'], forwards['forward_rate']*100, 
       linewidth=2, color='#A23B72', linestyle='--', alpha=0.7, label='Tasas Forward')

ax.set_xlabel('Plazo (años)', fontsize=12, fontweight='bold')
ax.set_ylabel('Tasa de Rendimiento (%)', fontsize=12, fontweight='bold')
ax.set_title('Yield Curve - Bonos Cupón Cero (Spot, Forward e Interpolación)', 
            fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=11, loc='best')
plt.tight_layout()
plt.show()

## 10. Análisis Completo

In [ ]:
# Análisis integral
analisis = calc.analyze(bonos_reales[['dias', 'precio']])

print("="*60)
print("ANÁLISIS COMPLETO DE YIELD CURVE")
print("="*60)

print(f"\n📋 Información General:")
print(f"   Número de bonos: {len(bonos_reales)}")
print(f"   Plazo mínimo: {analisis['min_days']/360:.2f} años ({analisis['min_days']} días)")
print(f"   Plazo máximo: {analisis['max_days']/360:.2f} años ({analisis['max_days']} días)")
print(f"   Nominal: 100")
print(f"   Tipo: Bonos cupón cero")

print(f"\n📊 Estadísticas de Tasas Spot:")
print(f"   Mínima: {analisis['min_yield']*100:.4f}%")
print(f"   Máxima: {analisis['max_yield']*100:.4f}%")
print(f"   Promedio: {analisis['avg_yield']*100:.4f}%")
print(f"   Rango: {(analisis['max_yield']-analisis['min_yield'])*100:.4f}%")

print(f"\n📈 Forma de la Curva:")
first_rate = analisis['yields_df']['yield'].iloc[0]
last_rate = analisis['yields_df']['yield'].iloc[-1]
if last_rate > first_rate:
    print(f"   Pendiente positiva (normal) ⬆️")
elif last_rate < first_rate:
    print(f"   Pendiente negativa (invertida) ⬇️")
else:
    print(f"   Pendiente plana ➡️")

print(f"\n✅ Análisis completado exitosamente")

## 11. Comparación: Spot vs Forward

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Tasas Spot
ax1.bar(bonos_reales['plazo_años'], yields['yield']*100, 
       color='#2E86AB', alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Plazo (años)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Tasa (%)', fontsize=11, fontweight='bold')
ax1.set_title('Tasas Spot (Z(0,T))', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Tasas Forward
ax2.bar(bonos_reales['plazo_años'], forwards['forward_rate']*100, 
       color='#A23B72', alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Plazo (años)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Tasa (%)', fontsize=11, fontweight='bold')
ax2.set_title('Tasas Forward f(t0,t1)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 12. Exportar Resultados

In [ ]:
# Crear DataFrame con resultados completos
resultados = pd.DataFrame({
    'plazo_años': bonos_reales['plazo_años'],
    'plazo_días': bonos_reales['dias'],
    'precio': bonos_reales['precio'],
    'tasa_spot_%': yields['yield'] * 100,
    'tasa_forward_%': forwards['forward_rate'] * 100
})

print("\n📊 Resultados Finales:")
print("="*80)
print(resultados.round(4).to_string(index=False))

# Descargar como CSV
resultados.to_csv('yield_curve_resultados.csv', index=False)
print("\n✅ Resultados exportados a 'yield_curve_resultados.csv'")

## Resumen

En este análisis aprendiste a:

1. ✅ Cargar **datos reales de bonos cupón cero**
2. ✅ Calcular la **yield curve (tasas spot)**
3. ✅ Hacer **bootstrapping** para obtener tasas forward
4. ✅ **Interpolar** tasas para plazos intermedios
5. ✅ **Visualizar** la curva en múltiples formas
6. ✅ **Analizar** la forma y características de la curva
7. ✅ **Comparar** tasas spot vs forward
8. ✅ **Exportar** resultados

### Aplicaciones Prácticas:
- Valuación de bonos y derivados
- Análisis de riesgo de tasa de interés
- Estrategias de inversión
- Gestión de portafolios

Para más información: [bonos-lib en GitHub](https://github.com/Jorgepuszkar/bonos-lib)